# 판결문 RAG의 근거 검색 / Retrieving the source case

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/legal_rag_retrieval.ipynb)

Run all cells in a CPU notebook (Python 3.11+). The 294 MB snapshot comes from the [public Korean precedent dataset](https://huggingface.co/datasets/woalsdnd/law.go.kr), published with an MIT license by its maintainer and derived from the National Law Information Center. This experiment uses no AI Hub files. Set the environment variable `CHART_LANG` to `ko` (default), `en` or `both` before running the plot.

## Imports

In [ ]:
import hashlib
import os
import platform
import random
import re
import shutil
import tempfile
import zipfile
from pathlib import Path
from urllib.request import urlopen

import matplotlib.pyplot as plt
import numpy as np
import sklearn
from matplotlib import font_manager
from sklearn.feature_extraction.text import TfidfVectorizer

## Fetch the pinned source. The publisher marks the dataset MIT.

In [ ]:
# Source: Korean Ministry of Government Legislation, collected by jaeminSon.
# https://huggingface.co/datasets/woalsdnd/law.go.kr
revision = '972b50e77f89b2d85f5fa950030e44a50861003e'
url = f'https://huggingface.co/datasets/woalsdnd/law.go.kr/resolve/{revision}/crawled_data.zip'
digest = '7147a31cea7d0206da0cad0d830fbb6dd81b0c83bc539cf0c2f987789e798b2b'
cache = Path('data/legal-rag-retrieval')
cache.mkdir(parents=True, exist_ok=True)
archive = cache / f'public-cases-{revision[:12]}.zip'

def checksum(path):
    with path.open('rb') as handle:
        return hashlib.file_digest(handle, 'sha256').hexdigest()

if archive.exists():
    if checksum(archive) != digest:
        raise ValueError(f'Unexpected file at {archive}. Move it aside, then rerun.')
else:
    with tempfile.TemporaryDirectory(dir=cache) as scratch:
        download = Path(scratch) / 'download.zip'
        with urlopen(url, timeout=120) as response, download.open('wb') as out:
            shutil.copyfileobj(response, out)
        if checksum(download) != digest:
            raise ValueError('Archive checksum mismatch. Retry the download.')
        download.replace(archive)

## Pair independently stored sections; do not index the query, party names or case ID.

In [ ]:
section_mark = re.compile(r'【([^】\n]+)】')
def normalize(value):
    value = re.sub(r'\[\d+\]', ' ', value)
    return ' '.join(value.split())

cases = []
used_questions, used_summaries = set(), set()
with zipfile.ZipFile(archive) as corpus:
    names = sorted(name for name in corpus.namelist() if re.fullmatch(r'crawled_data/licPrec\d+\.txt', name))
    random.Random(20261125).shuffle(names)
    for name in names:
        pieces = section_mark.split(corpus.read(name).decode('utf-8'))
        sections = dict(zip(pieces[1::2], pieces[2::2]))
        question = normalize(sections.get('판시사항', ''))[:280].strip()
        summary = normalize(sections.get('판결요지', ''))[:1800].strip()
        if len(question) < 20 or len(summary) < 60:
            continue
        if question in used_questions or summary in used_summaries:
            continue
        used_questions.add(question)
        used_summaries.add(summary)
        identifier = int(re.search(r'(\d+)\.txt$', name)[1])
        cases.append((identifier, question, summary))
        if len(cases) == 1000:
            break
assert len(cases) == 1000
cases.sort()  # Stable tie-breaker: increasing record ID, never query order preference.
ids, questions, summaries = zip(*cases)
print(f'Paired records: {len(cases)} | Python {platform.python_version()} | sklearn {sklearn.__version__}')

## Fit IDF on the indexed summaries alone. Rank with cosine on L2-normalized vectors.

In [ ]:
cutoffs = [1, 3, 5, 10]
settings = {
    'word': dict(analyzer='word', token_pattern=r'(?u)\b\w+\b'),
    'char': dict(analyzer='char', ngram_range=(2, 3)),
}
ranks = {}
first_hits = {}
for method, options in settings.items():
    vectorizer = TfidfVectorizer(dtype=np.float64, **options)
    docs = vectorizer.fit_transform(summaries)
    queries = vectorizer.transform(questions)
    scores = (queries @ docs.T).toarray()
    ranking = np.argsort(-scores, axis=1, kind='stable')
    positions = np.argmax(ranking == np.arange(len(cases))[:, None], axis=1) + 1
    # The source case must have positive similarity to count as retrieved.
    positions[np.diag(scores) <= 0] = len(cases) + 1
    ranks[method] = positions
    first_hits[method] = ranking[0, :3]

for method, positions in ranks.items():
    print(method, ' | '.join(f'R@{k}: {(positions <= k).sum()}/{len(cases)}' for k in cutoffs))
print('R@5 overlap | both, word only, char only, neither')
word_hit, char_hit = ranks['word'] <= 5, ranks['char'] <= 5
print(*(int(mask.sum()) for mask in [word_hit & char_hit, word_hit & ~char_hit,
                                    ~word_hit & char_hit, ~word_hit & ~char_hit]))

## Build a traceable context packet; case ID is attached only after retrieval.

In [ ]:
def context_packet(indices):
    return [{'source': f'https://www.law.go.kr/LSW/precInfoP.do?precSeq={ids[i]}',
             'summary': summaries[i]} for i in indices]

packet = context_packet(first_hits['char'])
assert all(item['summary'] and item['source'] for item in packet)
print('Example query source:', ids[0], '| retrieved:', [ids[i] for i in first_hits['char']])

## Plot in the requested language, using the same observations for both editions.

In [ ]:
language = os.environ.get('CHART_LANG', 'ko')
if language not in {'ko', 'en', 'both'}:
    raise ValueError('CHART_LANG must be ko, en or both')
languages = ['ko', 'en'] if language == 'both' else [language]
if 'ko' in languages:
    # Google Fonts, NanumGothic, SIL Open Font License 1.1.
    font = Path('NanumGothic-Regular.ttf')
    if not font.exists():
        with urlopen('https://raw.githubusercontent.com/google/fonts/main/ofl/nanumgothic/NanumGothic-Regular.ttf', timeout=60) as src:
            font.write_bytes(src.read())
    font_manager.fontManager.addfont(str(font))
    plt.rcParams['font.family'] = font_manager.FontProperties(fname=str(font)).get_name()
for lang in languages:
    ko = lang == 'ko'
    fig, ax = plt.subplots(figsize=(8.3, 4.5), dpi=190, layout='constrained')
    labels = ['단어', '글자 2~3개'] if ko else ['Word', 'Character 2–3']
    for (method, positions), label, color in zip(ranks.items(), labels, ['#276786', '#b45521']):
        ax.plot(cutoffs, [(positions <= k).mean() * 100 for k in cutoffs],
                color=color, marker='o', linewidth=2.5, label=label)
    ax.set(xticks=cutoffs, ylim=(0, 100),
           xlabel='후보 수 k' if ko else 'Retrieved cases (k)',
           ylabel='원 판결 회수율 (%)' if ko else 'Source-case recall (%)')
    ax.set_title('판시사항으로 원 판결의 요지 찾기' if ko else 'Recovering a source case from its issue statement', loc='left')
    ax.spines[['top','right']].set_visible(False)
    ax.grid(axis='y', alpha=.22)
    ax.legend(frameon=False)
    plt.show()